# 01 — Exploratory Data Analysis (EDA)

This notebook performs a comprehensive exploratory data analysis of the **COCO val2017** dataset,
focusing on **object density per image** — the core challenge in high-density segmentation.

We will examine:
1. Overall dataset statistics
2. How object counts are distributed across images
3. Which categories dominate the dataset
4. Visual samples at various density levels
5. Key challenges that will affect our baseline models

---
## Section 1 — Dataset Overview

We begin by loading the COCO annotations and computing basic statistics.
The COCO val2017 split contains 5,000 images with annotations for 80 object categories.
Understanding the overall scale of the dataset is essential before diving into density analysis.

In [1]:
import sys, os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from PIL import Image
from collections import Counter
from tqdm import tqdm

# Add project root to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.data_loader import get_dense_images, load_image_and_masks, get_dataset_stats

# Ensure output directories exist
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load COCO annotations
ann_file = '../data/annotations/annotations/instances_val2017.json'
print(f'Loading annotations from {ann_file} ...')
coco = COCO(ann_file)
print()

# Get dataset statistics
stats = get_dataset_stats(coco)
print()
print(f'Number of unique categories: {stats["num_categories"]}')

Loading annotations from ../data/annotations/annotations/instances_val2017.json ...
loading annotations into memory...


Done (t=0.40s)


creating index...
index created!

Total images       : 5000
Total annotations  : 36335
Unique categories  : 80
Avg objects/image  : 7.27
Min objects/image  : 0
Max objects/image  : 62

Number of unique categories: 80


---
## Section 2 — Object Density Distribution

Understanding how many objects appear per image is critical for high-density segmentation.
Images with more objects are harder to segment because of **occlusion** (objects blocking each other)
and **crowding** (many small objects packed tightly together).

We create two visualisations:
- A **histogram** showing the full distribution of object counts per image
- A **bar chart** grouping images into four density bands: 1–5, 5–15, 15–30, and 30–50 objects

In [3]:
# Compute object counts per image
img_ids = coco.getImgIds()
obj_counts = []
for img_id in tqdm(img_ids, desc='Counting objects per image'):
    ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=False)
    obj_counts.append(len(ann_ids))

obj_counts = np.array(obj_counts)
print(f'Computed counts for {len(obj_counts)} images.')
print(f'Mean: {obj_counts.mean():.2f}, Median: {np.median(obj_counts):.1f}, '
      f'Std: {obj_counts.std():.2f}')

Counting objects per ima

Counting objects per ima

Computed counts for 5000 images.
Mean: 7.27, Median: 4.0, Std: 7.25


In [4]:
# Figure: Histogram + Density Group Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Histogram ---
axes[0].hist(obj_counts, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Number of Objects per Image', fontsize=12)
axes[0].set_ylabel('Number of Images', fontsize=12)
axes[0].set_title('Distribution of Object Counts per Image', fontsize=14, fontweight='bold')
axes[0].axvline(obj_counts.mean(), color='red', linestyle='--', label=f'Mean = {obj_counts.mean():.1f}')
axes[0].legend(fontsize=11)

# --- Density groups bar chart ---
groups = {
    '1–5 (Low)': np.sum((obj_counts >= 1) & (obj_counts <= 5)),
    '5–15 (Medium)': np.sum((obj_counts > 5) & (obj_counts <= 15)),
    '15–30 (High)': np.sum((obj_counts > 15) & (obj_counts <= 30)),
    '30–50 (Very High)': np.sum((obj_counts > 30) & (obj_counts <= 50)),
}
colors = ['#4CAF50', '#FF9800', '#F44336', '#9C27B0']
bars = axes[1].bar(groups.keys(), groups.values(), color=colors, edgecolor='white')
axes[1].set_xlabel('Density Group', fontsize=12)
axes[1].set_ylabel('Number of Images', fontsize=12)
axes[1].set_title('Images by Object Density Group', fontsize=14, fontweight='bold')
for bar, val in zip(bars, groups.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                str(val), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/density_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/density_distribution.png')

# Print group counts
for name, count in groups.items():
    print(f'  {name}: {count} images')

Saved: results/figures/density_distribution.png
  1–5 (Low): 2769 images
  5–15 (Medium): 1541 images
  15–30 (High): 577 images
  30–50 (Very High): 60 images


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_59504/773364726.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 3 — Top Categories

COCO has 80 object categories, but they are not equally represented.
Some categories like **person** appear thousands of times, while others like **toaster** are rare.
This **class imbalance** will make it harder for models to learn to segment rare objects.

We plot the top 20 most frequent categories to visualise this imbalance.

In [5]:
# Count annotations per category
cat_ids = coco.getCatIds()
cats = coco.loadCats(cat_ids)
cat_names = {cat['id']: cat['name'] for cat in cats}

ann_ids_all = coco.getAnnIds(iscrowd=False)
anns_all = coco.loadAnns(ann_ids_all)

cat_counter = Counter()
for ann in tqdm(anns_all, desc='Counting categories'):
    cat_counter[cat_names[ann['category_id']]] += 1

top20 = cat_counter.most_common(20)
names = [item[0] for item in top20]
counts = [item[1] for item in top20]

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(names[::-1], counts[::-1], color='#2196F3', edgecolor='white')
ax.set_xlabel('Number of Annotations', fontsize=12)
ax.set_title('Top 20 Most Frequent Object Categories (COCO val2017)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/category_distribution.png')

Counting categories:   0

Counting categories: 100

Saved: results/figures/category_distribution.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_59504/2870369136.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 4 — Visual Samples

Seeing actual images at different density levels gives us an intuitive understanding of the challenge.
We select 6 images:
- **2 low-density** (1–5 objects): relatively easy with separated objects
- **2 medium-density** (10–20 objects): moderate overlap and crowding
- **2 high-density** (30+ objects): severe occlusion and many tiny objects

For each image we overlay the COCO ground-truth segmentation masks to show the target output.

In [6]:
import random
random.seed(42)

img_dir = '../data/images/val2017'

# Classify images into density groups
low_ids = [img_id for img_id, c in zip(img_ids, obj_counts) if 1 <= c <= 5]
med_ids = [img_id for img_id, c in zip(img_ids, obj_counts) if 10 <= c <= 20]
hi_ids  = [img_id for img_id, c in zip(img_ids, obj_counts) if c >= 30]

print(f'Low density (1-5):    {len(low_ids)} images')
print(f'Medium density (10-20): {len(med_ids)} images')
print(f'High density (30+):  {len(hi_ids)} images')

# Pick 2 from each group
sample_low = random.sample(low_ids, 2)
sample_med = random.sample(med_ids, 2)
sample_hi  = random.sample(hi_ids, 2)
sample_ids = sample_low + sample_med + sample_hi
labels = ['Low', 'Low', 'Medium', 'Medium', 'High', 'High']

print(f'\nSelected image IDs: {sample_ids}')

Low density (1-5):    2769 images
Medium density (10-20): 1014 images
High density (30+):  71 images

Selected image IDs: [299609, 476215, 559842, 474293, 316666, 546219]


In [7]:
# Visualise samples with overlaid masks
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (img_id, label) in enumerate(zip(sample_ids, labels)):
    ax = axes[idx // 3][idx % 3]
    image, anns = load_image_and_masks(coco, img_id, img_dir)
    ax.imshow(image)
    # Overlay masks
    for ann in anns:
        if 'segmentation' in ann and not ann.get('iscrowd', False):
            color = np.random.random(3)
            if isinstance(ann['segmentation'], list):
                for seg in ann['segmentation']:
                    poly = np.array(seg).reshape(-1, 2)
                    ax.fill(poly[:, 0], poly[:, 1], alpha=0.3, color=color)
                    ax.plot(poly[:, 0], poly[:, 1], linewidth=1, color=color)
    ax.set_title(f'{label} density — {len(anns)} objects (ID: {img_id})',
                 fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Images at Different Density Levels (with COCO masks)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/sample_images.png')

Saved: results/figures/sample_images.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_59504/2558205813.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 5 — Key Observations and Challenges

Based on our exploratory analysis, we identify **four specific challenges** that will
directly impact the performance of our baseline segmentation models:

### 1. Heavy Occlusion Between Objects
In high-density images (30+ objects), objects frequently overlap each other.
Occluded boundaries are ambiguous — the model cannot see where one object ends
and another begins. Classical methods like Watershed rely on clear intensity
gradients between objects, which disappear under occlusion. This will cause
the Watershed baseline to **under-segment** (merge adjacent objects into one region).

### 2. Scale Variation (Tiny vs Large Objects in the Same Image)
COCO images often contain objects that span a huge range of sizes — from a tiny
fork on a table to the full table itself. Fixed-parameter methods (e.g. a single
Gaussian blur kernel) cannot adapt to this variation. Small objects will be
smoothed away by the blur, while large objects may be over-segmented. This is a
fundamental limitation of methods that lack **multi-scale reasoning**.

### 3. Class Imbalance Across Categories
The category distribution is extremely skewed: **person** alone accounts for far
more annotations than the bottom 50 categories combined. While our baselines
are category-agnostic, this imbalance means that evaluation metrics will be
dominated by a few common classes. Any learning-based approach in Phase 2 will
need to address this through sampling strategies or loss weighting.

### 4. Objects Cut Off at Image Boundaries
Many annotations in COCO are for objects that are only partially visible —
they extend beyond the image frame. These truncated objects produce incomplete
masks that violate the assumptions of region-growing methods. Connected-component
analysis will treat boundary objects as whole regions, leading to incorrect
counts and oversized bounding regions.

---
**Summary:** These four challenges — occlusion, scale variation, class imbalance,
and boundary truncation — collectively explain why classical (non-learning) methods
are expected to perform poorly on dense COCO images. Our baseline experiments in
Notebook 02 will quantify this performance gap.

---
## Save Statistics

We save all computed statistics to a JSON file for reference in the report.

In [8]:
# Save all EDA stats to JSON
eda_stats = {
    'total_images': stats['total_images'],
    'total_annotations': stats['total_annotations'],
    'num_categories': stats['num_categories'],
    'avg_objects_per_image': round(stats['avg_objects'], 2),
    'min_objects_per_image': stats['min_objects'],
    'max_objects_per_image': stats['max_objects'],
    'density_groups': {
        '1-5_low': int(np.sum((obj_counts >= 1) & (obj_counts <= 5))),
        '5-15_medium': int(np.sum((obj_counts > 5) & (obj_counts <= 15))),
        '15-30_high': int(np.sum((obj_counts > 15) & (obj_counts <= 30))),
        '30-50_very_high': int(np.sum((obj_counts > 30) & (obj_counts <= 50))),
    },
    'top_10_categories': dict(Counter({cat_names[ann['category_id']]: 0 for ann in anns_all})),
}

# Fix top 10 — take actual top 10
eda_stats['top_10_categories'] = dict(cat_counter.most_common(10))

with open('../results/metrics/eda_stats.json', 'w') as f:
    json.dump(eda_stats, f, indent=2)

print('Saved: results/metrics/eda_stats.json')
print(json.dumps(eda_stats, indent=2))

Saved: results/metrics/eda_stats.json
{
  "total_images": 5000,
  "total_annotations": 36335,
  "num_categories": 80,
  "avg_objects_per_image": 7.27,
  "min_objects_per_image": 0,
  "max_objects_per_image": 62,
  "density_groups": {
    "1-5_low": 2769,
    "5-15_medium": 1541,
    "15-30_high": 577,
    "30-50_very_high": 60
  },
  "top_10_categories": {
    "person": 10777,
    "car": 1918,
    "chair": 1771,
    "book": 1129,
    "bottle": 1013,
    "cup": 895,
    "dining table": 695,
    "traffic light": 634,
    "bowl": 623,
    "handbag": 540
  }
}
